# Kandinsky Game→Audio — Step 9: Wire Event Tokens Through the Pipeline API

Step 8 passed: the PiFlow sampler forwarded game tokens on each denoising step in the tiny synthetic setup. This step extends the scratch patch to the public `Kandinsky6Pipeline.__call__` boundary and both sampler choices: Euler and PiFlow.

The proposed caller input is a numeric event-token tensor `[batch, events, game_dim]` plus an optional validity mask `[batch, events]`. The pipeline forwards these to the sampler. Event encoding itself is intentionally outside Kandinsky: this notebook does not define a universal game-engine schema or encoder.

## What a pass means

- The public pipeline method accepts optional event tokens and a validity mask.
- Pipeline validation checks generated-audio mode, batch size, event width, and mask shape.
- Both Euler and PiFlow receive the condition, then forward it into the DiT on every denoising call.
- The patch is confined to scratch copies; the pinned Kandinsky checkout stays unchanged.

**This is a source/API wiring audit only. It does not load weights, run a real pipeline, train, decode audio, or establish sound quality.**

## 1. Runtime

In [ ]:
import ast
import subprocess
from pathlib import Path

print("This notebook uses only Python's standard library.")
print("No GPU, model weights, or API credentials are needed.")

## 2. Fetch the exact Kandinsky source revision

In [ ]:
K6_COMMIT = "7ceeefad7fb625011f7db602c43893a989f19c15"
K6_DIR = Path("/content/kandinsky-6-step9-lab")

if not K6_DIR.exists():
    subprocess.run([
        "git", "clone", "--depth", "1", "--branch", "main",
        "https://github.com/kandinskylab/kandinsky-6.git", str(K6_DIR)
    ], check=True)

current = subprocess.check_output(
    ["git", "-C", str(K6_DIR), "rev-parse", "HEAD"], text=True
).strip()
if current != K6_COMMIT:
    subprocess.run(
        ["git", "-C", str(K6_DIR), "fetch", "--depth", "1", "origin", K6_COMMIT],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(K6_DIR), "checkout", "--detach", K6_COMMIT],
        check=True,
    )

assert subprocess.check_output(
    ["git", "-C", str(K6_DIR), "rev-parse", "HEAD"], text=True
).strip() == K6_COMMIT

DIT_PATH = K6_DIR / "kandinsky/core/components/dit.py"
EULER_PATH = K6_DIR / "kandinsky/core/algo/denoise_loop.py"
PIFLOW_PATH = K6_DIR / "kandinsky/core/algo/piflow_sampler.py"
PIPELINE_PATH = K6_DIR / "kandinsky/pipeline/pipeline.py"

originals = {
    "dit": DIT_PATH.read_text(encoding="utf-8"),
    "euler": EULER_PATH.read_text(encoding="utf-8"),
    "piflow": PIFLOW_PATH.read_text(encoding="utf-8"),
    "pipeline": PIPELINE_PATH.read_text(encoding="utf-8"),
}
source = originals["dit"]  # consumed by the shared full-DiT patch in the next section
print("Pinned source:", K6_COMMIT)
print("Pipeline:", PIPELINE_PATH)

## 3. Add the Game→Audio branch to a scratch DiT copy

This is the same full-DiT patch used in the earlier architecture steps. It creates no event encoder and loads no pretrained checkpoint.

In [ ]:
def replace_once(text, old, new, label):
    count = text.count(old)
    if count != 1:
        raise RuntimeError(f"Expected one source anchor for {label}; found {count}.")
    return text.replace(old, new, 1)

patched = source

# Add the optional event-token width to the real fused block.
patched = replace_once(
    patched,
    "        ca_rope: bool = False, cross_gates: bool = False, fix_modulation: bool = False,\n    ):\n",
    "        ca_rope: bool = False, cross_gates: bool = False, fix_modulation: bool = False,\n"
    "        game_dim: int | None = None,\n    ):\n",
    "fused-block constructor",
)

# Create the Game→Audio attention branch only when game_dim is configured.
patched = replace_once(
    patched,
    "        self.model_dim     = model_dim\n        self.model_dim_a   = model_dim_a\n",
    "        self.model_dim     = model_dim\n        self.model_dim_a   = model_dim_a\n"
    "        self.game_dim      = game_dim\n"
    "        if game_dim is None:\n"
    "            self.game_audio_norm = None\n"
    "            self.game_to_audio = None\n"
    "            self.game_audio_modulation = None\n"
    "        else:\n"
    "            self.game_audio_norm = nn.LayerNorm(model_dim_a, elementwise_affine=False)\n"
    "            self.game_to_audio = MultiheadCrossAttention(model_dim_a, head_dim_a, game_dim, engine)\n"
    "            self.game_audio_modulation = Modulation(time_dim_a, model_dim_a, 1)\n",
    "fused-block Game→Audio modules",
)

# Extend the fused block call. These are also used for audio-only calls.
patched = replace_once(
    patched,
    "        av_gate_scale: float = 1.0,\n"
    "        va_gate_scale: float = 1.0,\n"
    "    ):\n",
    "        av_gate_scale: float = 1.0,\n"
    "        va_gate_scale: float = 1.0,\n"
    "        game_tokens: Tensor | None = None,\n"
    "        game_attn_mask: Tensor | None = None,\n"
    "    ):\n",
    "fused-block forward signature",
)

# Apply the branch after existing text/AV updates and before the audio FFN.
patched = replace_once(
    patched,
    "        else:\n"
    "            vis = apply_gate_sum(vis, vis_out_t, gate_v).type_as(vis)\n"
    "\n"
    "        # ---- FFN ----\n",
    "        else:\n"
    "            vis = apply_gate_sum(vis, vis_out_t, gate_v).type_as(vis)\n"
    "\n"
    "        # Game events condition the audio stream in both audio-only and AV calls.\n"
    "        if game_tokens is not None:\n"
    "            if aud is None:\n"
    "                raise ValueError(\"Game conditioning requires an audio stream.\")\n"
    "            if self.game_to_audio is None:\n"
    "                raise ValueError(\"Construct this block with game_dim to accept game_tokens.\")\n"
    "            if game_tokens.ndim != 3 or game_tokens.shape[-1] != self.game_dim:\n"
    "                raise ValueError(\"game_tokens must have shape [batch, tokens, game_dim].\")\n"
    "            if game_tokens.shape[0] != aud.shape[0]:\n"
    "                raise ValueError(\"game_tokens and audio must have the same batch size.\")\n"
    "            game_tokens = game_tokens.to(\n"
    "                device=aud.device, dtype=self.game_to_audio.to_key.weight.dtype\n"
    "            )\n"
    "            if game_attn_mask is not None:\n"
    "                game_attn_mask = game_attn_mask.to(device=aud.device, dtype=torch.bool)\n"
    "            game_query = self.game_audio_norm(aud).type_as(aud)\n"
    "            game_update = self.game_to_audio(game_query, game_tokens, game_attn_mask)\n"
    "            game_gate = self.game_audio_modulation(t_a)\n"
    "            aud = apply_gate_sum(aud, game_update, game_gate).type_as(aud)\n"
    "\n"
    "        # ---- FFN ----\n",
    "fused-block Game→Audio insertion",
)

# Add game_dim to DiffusionTransformer3D and pass it to every fused block.
patched = replace_once(
    patched,
    "        visual_token_type_num_embeddings: int = 0,\n"
    "    ):\n",
    "        visual_token_type_num_embeddings: int = 0,\n"
    "        game_dim: int | None = None,\n"
    "    ):\n",
    "DiT constructor signature",
)
patched = replace_once(
    patched,
    "        self.is_multimodal  = is_multimodal\n"
    "        self.in_visual_dim  = in_visual_dim\n",
    "        self.is_multimodal  = is_multimodal\n"
    "        self.game_dim       = game_dim\n"
    "        if game_dim is not None and not is_multimodal:\n"
    "            raise ValueError(\"game_dim is supported only by the multimodal audio model.\")\n"
    "        self.in_visual_dim  = in_visual_dim\n",
    "DiT game_dim configuration",
)
patched = replace_once(
    patched,
    "                    ca_rope=ca_rope, cross_gates=cross_gates, fix_modulation=fix_modulation,\n",
    "                    ca_rope=ca_rope, cross_gates=cross_gates, fix_modulation=fix_modulation,\n"
    "                    game_dim=game_dim,\n",
    "pass game_dim to fused-block stack",
)

# Thread game tokens through the audio-only fused-block loop.
patched = replace_once(
    patched,
    "        aud_rope: Tensor | None,\n"
    "        attn_mask: Tensor | None = None,\n"
    "    ) -> tuple[Tensor | None, Tensor | None]:\n",
    "        aud_rope: Tensor | None,\n"
    "        attn_mask: Tensor | None = None,\n"
    "        game_tokens: Tensor | None = None,\n"
    "        game_attn_mask: Tensor | None = None,\n"
    "    ) -> tuple[Tensor | None, Tensor | None]:\n",
    "single-modality block-loop signature",
)
patched = replace_once(
    patched,
    "                    _, aud_embed = blk(\n"
    "                        None, aud_embed, te, te, (tm, tm), None, aud_rope, attn_mask,\n"
    "                    )\n",
    "                    _, aud_embed = blk(\n"
    "                        None, aud_embed, te, te, (tm, tm), None, aud_rope, attn_mask,\n"
    "                        game_tokens=game_tokens, game_attn_mask=game_attn_mask,\n"
    "                    )\n",
    "audio-only fused-block call",
)

# Thread the same tensors through the joint video+audio loop.
patched = replace_once(
    patched,
    "        aud_rope: Tensor,\n"
    "        attn_mask: Tensor | None = None,\n"
    "    ) -> tuple[Tensor, Tensor]:\n",
    "        aud_rope: Tensor,\n"
    "        attn_mask: Tensor | None = None,\n"
    "        game_tokens: Tensor | None = None,\n"
    "        game_attn_mask: Tensor | None = None,\n"
    "    ) -> tuple[Tensor, Tensor]:\n",
    "fused block-loop signature",
)
patched = replace_once(
    patched,
    "                vis_rope, aud_rope,\n"
    "                attn_mask,\n"
    "            )\n"
    "        return vis_embed, aud_embed\n",
    "                vis_rope, aud_rope,\n"
    "                attn_mask,\n"
    "                game_tokens=game_tokens, game_attn_mask=game_attn_mask,\n"
    "            )\n"
    "        return vis_embed, aud_embed\n",
    "joint fused-block call",
)

# Add public forward arguments and validate their basic contract.
patched = replace_once(
    patched,
    "        attention_mask: Tensor | None = None,\n"
    "        visual_token_type_ids: Tensor | None = None,\n"
    "    ) -> Tensor | tuple[Tensor, Tensor]:\n",
    "        attention_mask: Tensor | None = None,\n"
    "        visual_token_type_ids: Tensor | None = None,\n"
    "        game_tokens: Tensor | None = None,\n"
    "        game_attn_mask: Tensor | None = None,\n"
    "    ) -> Tensor | tuple[Tensor, Tensor]:\n",
    "public forward signature",
)
patched = replace_once(
    patched,
    "        both = x_video is not None and x_audio is not None and self.is_multimodal\n"
    "        attn_mask = self._normalize_attn_mask(attention_mask)\n",
    "        both = x_video is not None and x_audio is not None and self.is_multimodal\n"
    "        if game_attn_mask is not None and game_tokens is None:\n"
    "            raise ValueError(\"game_attn_mask requires game_tokens.\")\n"
    "        if game_tokens is not None:\n"
    "            if not self.is_multimodal or self.game_dim is None:\n"
    "                raise ValueError(\"Configure game_dim on a multimodal model before passing game_tokens.\")\n"
    "            if x_audio is None:\n"
    "                raise ValueError(\"Game conditioning requires x_audio.\")\n"
    "            if game_tokens.ndim != 3 or game_tokens.shape[-1] != self.game_dim:\n"
    "                raise ValueError(\"game_tokens must have shape [batch, events, game_dim].\")\n"
    "            audio_batch = x_audio.shape[0] if x_audio.ndim == 3 else 1\n"
    "            if game_tokens.shape[0] != audio_batch:\n"
    "                raise ValueError(\"game_tokens and x_audio must have the same batch size.\")\n"
    "            if game_attn_mask is not None:\n"
    "                if game_attn_mask.dtype != torch.bool:\n"
    "                    raise ValueError(\"game_attn_mask must be boolean; True means a valid event token.\")\n"
    "                if game_attn_mask.shape[-1] != game_tokens.shape[1]:\n"
    "                    raise ValueError(\"The final game_attn_mask dimension must match the event count.\")\n"
    "        attn_mask = self._normalize_attn_mask(attention_mask)\n",
    "forward argument validation",
)

# Pass the optional condition to audio-only and joint forward paths.
patched = replace_once(
    patched,
    "                None, aud_embed, te, tm, None, aud_rope, attn_mask,\n"
    "            )\n"
    "            return self._project_audio(aud_embed, tm)\n",
    "                None, aud_embed, te, tm, None, aud_rope, attn_mask,\n"
    "                game_tokens=game_tokens, game_attn_mask=game_attn_mask,\n"
    "            )\n"
    "            return self._project_audio(aud_embed, tm)\n",
    "audio-only DiT call",
)
patched = replace_once(
    patched,
    "            vis_rope, aud_rope, attn_mask,\n"
    "        )\n"
    "        return self._project_fused(\n",
    "            vis_rope, aud_rope, attn_mask,\n"
    "            game_tokens=game_tokens, game_attn_mask=game_attn_mask,\n"
    "        )\n"
    "        return self._project_fused(\n",
    "joint DiT call",
)

PATCH_PATH = DIT_PATH.with_name("dit_game_full.py")
PATCH_PATH.write_text(patched, encoding="utf-8")
print("Patched scratch copy:", PATCH_PATH)
print("Original source unchanged:", DIT_PATH.read_text(encoding="utf-8") == source)

In [ ]:
DIT_GAME_PATH = DIT_PATH.with_name("dit_game_full.py")
DIT_GAME_PATH.write_text(patched, encoding="utf-8")
print("Scratch DiT written:", DIT_GAME_PATH)

## 4. Extend both denoising loops

The DiT can now receive event tokens. We thread the same optional arguments through the ordinary Euler loop and the PiFlow loop so the selected sampler does not determine whether the event condition is available.

In [ ]:
euler_patched = replace_once(
    originals["euler"],
    "    visual_token_type_ids: Tensor | None = None,\n"
    "    recompute_ropes_each_step: bool = False,\n",
    "    visual_token_type_ids: Tensor | None = None,\n"
    "    game_tokens: Tensor | None = None,\n"
    "    game_attn_mask: Tensor | None = None,\n"
    "    recompute_ropes_each_step: bool = False,\n",
    "Euler sampler condition arguments",
)
euler_patched = replace_once(
    euler_patched,
    "                visual_token_type_ids=visual_token_type_ids,\n"
    "            )\n\n"
    "        vel_cond = _forward(\n",
    "                visual_token_type_ids=visual_token_type_ids,\n"
    "                game_tokens=game_tokens,\n"
    "                game_attn_mask=game_attn_mask,\n"
    "            )\n\n"
    "        vel_cond = _forward(\n",
    "Euler DiT forwarding",
)

piflow_patched = replace_once(
    originals["piflow"],
    "    visual_token_type_ids: torch.Tensor | None = None,\n"
    "    scale_factor: tuple[float, float, float] = (1.0, 1.0, 1.0),\n",
    "    visual_token_type_ids: torch.Tensor | None = None,\n"
    "    game_tokens: torch.Tensor | None = None,\n"
    "    game_attn_mask: torch.Tensor | None = None,\n"
    "    scale_factor: tuple[float, float, float] = (1.0, 1.0, 1.0),\n",
    "PiFlow sampler condition arguments",
)
piflow_patched = replace_once(
    piflow_patched,
    "            visual_token_type_ids=visual_token_type_ids,\n"
    "        )\n"
    "        pred_video = _normalize_grid_prediction",
    "            visual_token_type_ids=visual_token_type_ids,\n"
    "            game_tokens=game_tokens,\n"
    "            game_attn_mask=game_attn_mask,\n"
    "        )\n"
    "        pred_video = _normalize_grid_prediction",
    "PiFlow DiT forwarding",
)

EULER_GAME_PATH = EULER_PATH.with_name("denoise_loop_game.py")
PIFLOW_GAME_PATH = PIFLOW_PATH.with_name("piflow_sampler_game.py")
EULER_GAME_PATH.write_text(euler_patched, encoding="utf-8")
PIFLOW_GAME_PATH.write_text(piflow_patched, encoding="utf-8")
print("Scratch Euler sampler:", EULER_GAME_PATH)
print("Scratch PiFlow sampler:", PIFLOW_GAME_PATH)

## 5. Expose event tokens at the public pipeline call

The public API accepts a batched event tensor `[B, N, D]`. The optional mask uses the friendlier `[B, N]` shape; the pipeline expands it to the attention shape `[B, 1, 1, N]`. The model's attached adapter must declare the matching `game_dim`.

For this experiment, supplying event tokens is valid only when the pipeline will generate audio. Passing tokens while returning a supplied audio track would not test event-conditioned synthesis.

In [ ]:
pipeline_patched = originals["pipeline"]

pipeline_patched = replace_once(
    pipeline_patched,
    "from ..core.algo.denoise_loop import denoise_loop\n",
    "from ..core.algo.denoise_loop_game import denoise_loop\n",
    "pipeline Euler import",
)
pipeline_patched = replace_once(
    pipeline_patched,
    "from ..core.algo.piflow_sampler import piflow_denoise_loop\n",
    "from ..core.algo.piflow_sampler_game import piflow_denoise_loop\n",
    "pipeline PiFlow import",
)

pipeline_patched = replace_once(
    pipeline_patched,
    "        audio: str | Path | None = None,\n"
    "        video: str | Path | None = None,\n"
    "    ) -> Kandinsky6PipelineOutput:\n",
    "        audio: str | Path | None = None,\n"
    "        video: str | Path | None = None,\n"
    "        game_tokens: torch.Tensor | None = None,\n"
    "        game_attn_mask: torch.Tensor | None = None,\n"
    "    ) -> Kandinsky6PipelineOutput:\n",
    "public pipeline event arguments",
)

pipeline_patched = replace_once(
    pipeline_patched,
    "        generate_audio = do_audio and audio is None\n"
    "        generate_video = video is None\n",
    "        generate_audio = do_audio and audio is None\n"
    "        generate_video = video is None\n"
    "        if game_attn_mask is not None and game_tokens is None:\n"
    "            raise ValueError(\"game_attn_mask requires game_tokens\")\n"
    "        if game_tokens is not None:\n"
    "            game_tokens = torch.as_tensor(game_tokens)\n"
    "            if not generate_audio:\n"
    "                raise ValueError(\"game_tokens require generated audio output\")\n"
    "            if not getattr(dit, \"is_multimodal\", False):\n"
    "                raise ValueError(\"game_tokens require a multimodal audio model\")\n"
    "            game_dim = getattr(dit, \"game_dim\", None)\n"
    "            if game_dim is None:\n"
    "                raise ValueError(\"Attach a Game→Audio adapter before passing game_tokens\")\n"
    "            if game_tokens.ndim != 3 or game_tokens.shape[1] < 1 or game_tokens.shape[2] != game_dim:\n"
    "                raise ValueError(\"game_tokens must have shape [batch, events, game_dim]\")\n"
    "            if not game_tokens.is_floating_point():\n"
    "                raise ValueError(\"game_tokens must contain floating-point features\")\n"
    "            if game_tokens.shape[0] != batch_size:\n"
    "                raise ValueError(\"game_tokens batch size must match the pipeline request\")\n"
    "            game_tokens = game_tokens.to(device=self.device)\n"
    "            if game_attn_mask is not None:\n"
    "                game_attn_mask = torch.as_tensor(game_attn_mask)\n"
    "                if game_attn_mask.dtype != torch.bool:\n"
    "                    raise ValueError(\"game_attn_mask must be boolean; True means valid event\")\n"
    "                if tuple(game_attn_mask.shape) != tuple(game_tokens.shape[:2]):\n"
    "                    raise ValueError(\"game_attn_mask must have shape [batch, events]\")\n"
    "                if not game_attn_mask.any(dim=1).all():\n"
    "                    raise ValueError(\"each request needs at least one valid event token\")\n"
    "                game_attn_mask = game_attn_mask[:, None, None, :].to(device=self.device)\n",
    "pipeline event validation and device placement",
)

pipeline_patched = replace_once(
    pipeline_patched,
    "                        attention_mask=attention_mask,\n"
    "                        visual_token_type_ids=visual_token_type_ids,\n"
    "                        scale_factor=self.scale_factor,\n",
    "                        attention_mask=attention_mask,\n"
    "                        visual_token_type_ids=visual_token_type_ids,\n"
    "                        game_tokens=game_tokens,\n"
    "                        game_attn_mask=game_attn_mask,\n"
    "                        scale_factor=self.scale_factor,\n",
    "pipeline PiFlow condition call",
)
pipeline_patched = replace_once(
    pipeline_patched,
    "                        attention_mask=attention_mask,\n"
    "                        null_attention_mask=null_attention_mask,\n"
    "                        visual_token_type_ids=visual_token_type_ids,\n"
    "                        scale_factor=self.scale_factor,\n",
    "                        attention_mask=attention_mask,\n"
    "                        null_attention_mask=null_attention_mask,\n"
    "                        visual_token_type_ids=visual_token_type_ids,\n"
    "                        game_tokens=game_tokens,\n"
    "                        game_attn_mask=game_attn_mask,\n"
    "                        scale_factor=self.scale_factor,\n",
    "pipeline Euler condition call",
)

PIPELINE_GAME_PATH = PIPELINE_PATH.with_name("pipeline_game.py")
PIPELINE_GAME_PATH.write_text(pipeline_patched, encoding="utf-8")
print("Scratch pipeline written:", PIPELINE_GAME_PATH)
print("Original files still unchanged on disk:", all(
    path.read_text(encoding="utf-8") == originals[key]
    for key, path in {
        "dit": DIT_PATH,
        "euler": EULER_PATH,
        "piflow": PIFLOW_PATH,
        "pipeline": PIPELINE_PATH,
    }.items()
))

## 6. Audit the end-to-end source contract

We parse the patched source without importing Kandinsky or constructing its pipeline. This checks that the public signature, both pipeline branches, both sampler forwards, and the DiT signature all agree on the optional arguments.

In [ ]:
def function_node(tree, name, class_name=None):
    if class_name is not None:
        cls = next(n for n in tree.body if isinstance(n, ast.ClassDef) and n.name == class_name)
        return next(n for n in cls.body if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef)) and n.name == name)
    return next(n for n in tree.body if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef)) and n.name == name)

def call_nodes(tree, name):
    return [
        node for node in ast.walk(tree)
        if isinstance(node, ast.Call)
        and isinstance(node.func, ast.Name)
        and node.func.id == name
    ]

source_map = {
    "dit": patched,
    "euler": euler_patched,
    "piflow": piflow_patched,
    "pipeline": pipeline_patched,
}
trees = {name: ast.parse(text, filename=name) for name, text in source_map.items()}

pipeline_call = function_node(trees["pipeline"], "__call__", "Kandinsky6Pipeline")
public_args = {arg.arg for arg in pipeline_call.args.args + pipeline_call.args.kwonlyargs}
assert {"game_tokens", "game_attn_mask"} <= public_args

for sampler_name in ("denoise_loop", "piflow_denoise_loop"):
    calls = call_nodes(trees["pipeline"], sampler_name)
    assert len(calls) == 1, f"Expected one {sampler_name} call in pipeline"
    kwargs = {kw.arg for kw in calls[0].keywords}
    assert {"game_tokens", "game_attn_mask"} <= kwargs

for source_name, function_name in (("euler", "denoise_loop"), ("piflow", "piflow_denoise_loop")):
    sampler_fn = function_node(trees[source_name], function_name)
    sampler_args = {arg.arg for arg in sampler_fn.args.args + sampler_fn.args.kwonlyargs}
    assert {"game_tokens", "game_attn_mask"} <= sampler_args
    dit_calls = call_nodes(sampler_fn, "dit")
    assert dit_calls, f"No DiT call found in {function_name}"
    assert all({"game_tokens", "game_attn_mask"} <= {kw.arg for kw in call.keywords} for call in dit_calls)

dit_forward = function_node(trees["dit"], "forward", "DiffusionTransformer3D")
dit_args = {arg.arg for arg in dit_forward.args.args + dit_forward.args.kwonlyargs}
assert {"game_tokens", "game_attn_mask"} <= dit_args

print("Public pipeline accepts event tokens and mask: passed")
print("Euler pipeline→sampler→DiT forwarding: passed")
print("PiFlow pipeline→sampler→DiT forwarding: passed")
print("All patched Python sources parse: passed")
print("No Kandinsky package import or model weights were used.")

## What Step 9 proves — and does not prove

**Proves if all cells pass:** the scratch pipeline has a consistent optional event-token API and routes the tensors through either sampler into the patched DiT. The source patches are pinned, anchor-checked, syntax-parsed, and do not modify the original checkout.

**Does not prove:** that a real pretrained checkpoint loads with the sidecar, that the full pipeline executes with the patch, or that event tokens produce useful or audible control. No actual runtime call is made here.

**Next:** decide and implement the versioned engine-event encoder contract, then load an allowed checkpoint with its adapter and run one real video→audio generation through the patched pipeline. Training should wait until the event schema and paired supervision data are defined.